# Build an ICD-condition database

For every training note, the model extracts clinical conditions and then compares them with existing condition columns. The CSV has one row per gold ICD code and a `0` or `1` for each condition. If a note has multiple gold codes, its conditions are applied to each code; repeated codes accumulate conditions across notes.

**Running the cells sends note text to the configured model provider and incurs API charges.** This makes two requests per training note (100 requests for the current 50-note split). Progress is saved after each completed note. Rerunning starts over and overwrites `out/icd_conditions_train.csv`. Saved outputs from earlier runs may show the old full-data results; rerun the cells for this split.

In [10]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

root = Path.cwd()
if not (root / 'llm.py').is_file():
    raise FileNotFoundError('Start the notebook with the project root as its working directory.')

provider = 'openai'
model = None  # None uses the provider's MODEL environment variable or llm.py default
api_python = '/opt/homebrew/bin/python3'  # Python that can verify your network's TLS certificate
notes_path = root / 'data' / 'dev_notes_train.jsonl'
extract_prompt_path = root / 'prompts' / 'extract_conditions.txt'
match_prompt_path = root / 'prompts' / 'match_conditions.txt'
output_path = root / 'out' / 'icd_conditions_train.csv'

## Load notes and prompts

Edit the two prompt files to tune extraction and matching. Rerun this cell after editing them.

In [11]:
with notes_path.open(encoding='utf-8') as file:
    notes = [json.loads(line) for line in file if line.strip()]
for note in notes:
    if not isinstance(note, dict) or not isinstance(note.get('text'), str) or not isinstance(note.get('gold'), list) or not note['gold']:
        raise ValueError('Each note must have text and at least one gold code.')
    if any(not isinstance(code, str) or not code.strip() for code in note['gold']):
        raise ValueError(f"Invalid gold code in note {note.get('id', '<unknown>')}")
extract_prompt = extract_prompt_path.read_text(encoding='utf-8').strip()
match_prompt = match_prompt_path.read_text(encoding='utf-8').strip()
if not extract_prompt or not match_prompt:
    raise ValueError('Extraction and matching prompts must not be empty.')
print(f'Loaded {len(notes)} notes.')

Loaded 50 notes.


## Define the model calls

Requests use the existing `llm.py` API client through the configured Python executable. The API key is read from the environment or project `.env` and is not placed in the notebook or command-line arguments.

In [12]:
import condition_utils

def call_model(prompt, input_text):
    return condition_utils.call_model(prompt, input_text, root=root, provider=provider, model=model, api_python=api_python)

def extract_conditions(note_text):
    return condition_utils.extract_conditions(note_text, extract_prompt, call_model)

def match_conditions(conditions, columns):
    return condition_utils.match_conditions(conditions, columns, match_prompt, call_model)

## Build and save the table

Only fully processed notes appear in the saved CSV if a request fails. Rerunning this cell starts a new build from the first note. The model only matches condition names; Python assigns all 0/1 values.

In [13]:
output_path.parent.mkdir(parents=True, exist_ok=True)
feature_columns = []
code_conditions = {}
df = pd.DataFrame(columns=['icd_code'])

def build_binary_row(code, present_conditions, columns):
    return {'icd_code': code, **{column: int(column in present_conditions) for column in columns}}

def save_table():
    global df
    records = [
        build_binary_row(code, present, feature_columns)
        for code, present in code_conditions.items()
    ]
    df = pd.DataFrame.from_records(records, columns=['icd_code', *feature_columns])
    temporary_path = output_path.with_suffix('.csv.tmp')
    df.to_csv(temporary_path, index=False)
    temporary_path.replace(output_path)

save_table()
for index, note in enumerate(notes, 1):
    print(f"[{index}/{len(notes)}] {note.get('id', '<unknown>')}: extracting conditions", flush=True)
    conditions = extract_conditions(note['text'])
    print(f"[{index}/{len(notes)}] {note.get('id', '<unknown>')}: matching columns", flush=True)
    mappings = match_conditions(conditions, list(df.columns[1:]))
    note_columns = set()
    for condition in conditions:
        column = mappings[condition]
        if column is None:
            column = next((existing for existing in feature_columns if existing.casefold() == condition.casefold()), condition)
            if column.casefold() == 'icd_code':
                raise ValueError('A condition cannot use the reserved icd_code column name.')
            if column not in feature_columns:
                feature_columns.append(column)
        note_columns.add(column)
    for code in note['gold']:
        code_conditions.setdefault(code, set()).update(note_columns)
    save_table()
print(f'Saved {len(df)} ICD codes and {len(feature_columns)} condition columns to {output_path}')

[1/50] dev-001: extracting conditions
[1/50] dev-001: matching columns
[2/50] dev-002: extracting conditions
[2/50] dev-002: matching columns
[3/50] dev-003: extracting conditions
[3/50] dev-003: matching columns
[4/50] dev-004: extracting conditions
[4/50] dev-004: matching columns
[5/50] dev-005: extracting conditions
[5/50] dev-005: matching columns
[6/50] dev-006: extracting conditions
[6/50] dev-006: matching columns
[7/50] dev-007: extracting conditions
[7/50] dev-007: matching columns
[8/50] dev-008: extracting conditions
[8/50] dev-008: matching columns
[9/50] dev-009: extracting conditions
[9/50] dev-009: matching columns
[10/50] dev-010: extracting conditions
[10/50] dev-010: matching columns
[11/50] dev-011: extracting conditions
[11/50] dev-011: matching columns
[12/50] dev-012: extracting conditions
[12/50] dev-012: matching columns
[13/50] dev-013: extracting conditions
[13/50] dev-013: matching columns
[14/50] dev-014: extracting conditions
[14/50] dev-014: matching colu

## Inspect the resulting table

In [14]:
display(df)
print(f'CSV: {output_path}')

,icd_code,falciparum malaria,fever,chills,rigors,frontal headache,generalized body aches,poor appetite,mild splenomegaly,viral upper respiratory infection,...,proteinuria,blurred vision,pedal edema,hyperreflexia,anaphylaxis,peanut allergy,lip swelling,hypotension,atrial fibrillation,irregular heartbeat
0,1F40.Z,1,1,1,1,1,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
1,CA07.0,0,1,0,0,0,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,CA00,0,1,0,0,0,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
3,CA23.30,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1A07.Z,0,1,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
5,1F23.10,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,1E90.0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,CA40.Z,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,1G05,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,1B10.0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


CSV: /Users/shivangi.s.1/Desktop/sugen_test/out/icd_conditions_train.csv


In [15]:
df.shape

(51, 194)

In [16]:
len(df.icd_code.unique())



51